In [2]:
!pip install py-tgb


In [4]:
from tgb.linkproppred.dataset import LinkPropPredDataset

dataset = LinkPropPredDataset(name="tgbl-wiki", root="datasets", preprocess=True)
data = dataset.full_data


Dataset tgbl-wiki version 2 not found.
Please download the latest version of the dataset.


Will you download the dataset(s) now? (y/N)
 y


Download started, this might take a while . . . 
Dataset title: tgbl-wiki
Download completed 
Dataset directory is  /Users/krishtadigotla/tgb_env/lib/python3.10/site-packages/tgb/datasets/tgbl_wiki
file not processed, generating processed file


In [5]:
from tgb.linkproppred.dataset import LinkPropPredDataset

# Load the dataset (now that it is downloaded & processed)
dataset = LinkPropPredDataset(name="tgbl-wiki", root="datasets", preprocess=True)
data = dataset.full_data

print("Dataset loaded successfully:", dataset.name)
print("Available keys:", list(data.keys()))
print("Sources shape:", data["sources"].shape)
print("Destinations shape:", data["destinations"].shape)
print("Timestamps shape:", data["timestamps"].shape)

# Show the first 5 interaction edges (source → destination at time t)
print("\nFirst 5 interactions:")
for i in range(5):
    print(f"{data['sources'][i]} -> {data['destinations'][i]}   @   t = {data['timestamps'][i]}")


file found, skipping download
Dataset directory is  /Users/krishtadigotla/tgb_env/lib/python3.10/site-packages/tgb/datasets/tgbl_wiki
loading processed file
Dataset loaded successfully: tgbl-wiki
Available keys: ['sources', 'destinations', 'timestamps', 'edge_idxs', 'edge_feat', 'w', 'edge_label']
Sources shape: (157474,)
Destinations shape: (157474,)
Timestamps shape: (157474,)

First 5 interactions:
0 -> 8227   @   t = 0.0
1 -> 8228   @   t = 36.0
1 -> 8228   @   t = 77.0
2 -> 8229   @   t = 131.0
1 -> 8228   @   t = 150.0


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
from tgb.linkproppred.dataset import LinkPropPredDataset
import numpy as np

# Load dataset
dataset = LinkPropPredDataset(name="tgbl-wiki", root="datasets", preprocess=True)
data = dataset.full_data

sources = torch.tensor(data["sources"], dtype=torch.long)
destinations = torch.tensor(data["destinations"], dtype=torch.long)
timestamps = torch.tensor(data["timestamps"], dtype=torch.float)

# ---- Create temporal splits ----
num_edges = len(timestamps)
sorted_idx = torch.argsort(timestamps)  # sort by time

train_end = int(0.70 * num_edges)
val_end   = int(0.85 * num_edges)

train_idx = sorted_idx[:train_end]
valid_idx = sorted_idx[train_end:val_end]
test_idx  = sorted_idx[val_end:]

print("Train edges:", len(train_idx))
print("Val edges:", len(valid_idx))
print("Test edges:", len(test_idx))


class TemporalLinkDataset(Dataset):
    def __init__(self, idx):
        self.idx = idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        e = self.idx[i]
        return sources[e], destinations[e], timestamps[e]

train_data = TemporalLinkDataset(train_idx)
val_data   = TemporalLinkDataset(valid_idx)

train_loader = DataLoader(train_data, batch_size=512, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=1024, shuffle=False)

print("Data pipeline ready.")


file found, skipping download
Dataset directory is  /Users/krishtadigotla/tgb_env/lib/python3.10/site-packages/tgb/datasets/tgbl_wiki
loading processed file
Train edges: 110231
Val edges: 23621
Test edges: 23622
Data pipeline ready.


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TemporalLinkPredictor(nn.Module):
    def __init__(self, num_nodes, emb_dim=128, time_dim=32):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, emb_dim)

        # Time encoder: sinusoidal + linear projection
        self.time_linear = nn.Linear(time_dim, emb_dim)

        self.ff_layer = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )

        self.scorer = nn.CosineSimilarity(dim=-1)

        self.time_dim = time_dim
        self.emb_dim = emb_dim

    def encode_time(self, t):
        """
        Sinusoidal time encoding, continuous version
        """
        device = t.device
        t = t.unsqueeze(1)  # (batch, 1)
        freq = torch.exp(
            torch.arange(0, self.time_dim, 2, device=device).float() * (-math.log(10000.0) / self.time_dim)
        )
        sin = torch.sin(t * freq)
        cos = torch.cos(t * freq)
        time_feat = torch.cat([sin, cos], dim=-1)
        return self.time_linear(time_feat)  # project to emb_dim

    def forward(self, src, dst, t):
        h_src = self.node_emb(src)
        h_dst = self.node_emb(dst)

        t_enc = self.encode_time(t)

        # Update representation based on interaction & timestamp
        pair = torch.cat([h_src + t_enc, h_dst + t_enc], dim=-1)
        updated = self.ff_layer(pair)

        # Link score (higher = more likely)
        score = self.scorer(updated, h_dst)
        return score


In [13]:
num_nodes = max(sources.max(), destinations.max()).item() + 1
model = TemporalLinkPredictor(num_nodes=num_nodes).to("cpu")
print("Model initialized with", num_nodes, "nodes and embedding dim:", model.emb_dim)


Model initialized with 9227 nodes and embedding dim: 128


In [14]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from tqdm import tqdm
import numpy as np

device = "cpu"
model = model.to(device)

optimizer = Adam(model.parameters(), lr=1e-3)
epochs = 5  # Increase later — but this is good for testing

def negative_sample(batch_src, batch_dst):
    neg_dst = torch.randint(0, num_nodes, (len(batch_src),), device=device)
    return neg_dst

def compute_mrr(scores, labels):
    # ranks positives vs negatives (simplified MRR for custom model)
    sorted_idx = torch.argsort(scores, descending=True)
    rank = (sorted_idx == 0).nonzero(as_tuple=True)[0].item() + 1
    return 1.0 / rank

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for src, dst, ts in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        src, dst, ts = src.to(device), dst.to(device), ts.to(device)
        neg_dst = negative_sample(src, dst)

        pos_scores = model(src, dst, ts)
        neg_scores = model(src, neg_dst, ts)

        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        scores = torch.cat([pos_scores, neg_scores])

        loss = F.binary_cross_entropy_with_logits(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ---- VALIDATION ----
    model.eval()
    mrr_scores = []
    with torch.no_grad():
        for src, dst, ts in val_loader:
            src, dst, ts = src.to(device), dst.to(device), ts.to(device)
            neg_dst = negative_sample(src, dst)

            pos = model(src, dst, ts)
            neg = model(src, neg_dst, ts)

            batch_scores = torch.stack([pos, neg], dim=1)  # shape (batch, 2)
            batch_labels = torch.zeros_like(batch_scores)
            batch_labels[:, 0] = 1  # positive is first column

            # Compute MRR for each row
            for i in range(len(batch_scores)):
                mrr_scores.append(compute_mrr(batch_scores[i], batch_labels[i]))

    avg_mrr = np.mean(mrr_scores)
    print(f"Epoch {epoch+1} completed | Loss: {total_loss:.4f} | Val MRR: {avg_mrr:.4f}")

print("\nTraining complete.")


Epoch 1/5: 100%|██████████████████████████████| 216/216 [00:03<00:00, 62.96it/s]


Epoch 1 completed | Loss: 127.7516 | Val MRR: 0.9323


Epoch 2/5: 100%|██████████████████████████████| 216/216 [00:03<00:00, 70.82it/s]


Epoch 2 completed | Loss: 111.1376 | Val MRR: 0.9397


Epoch 3/5: 100%|██████████████████████████████| 216/216 [00:03<00:00, 70.53it/s]


Epoch 3 completed | Loss: 103.9221 | Val MRR: 0.9380


Epoch 4/5: 100%|██████████████████████████████| 216/216 [00:02<00:00, 76.44it/s]


Epoch 4 completed | Loss: 98.5147 | Val MRR: 0.9392


Epoch 5/5: 100%|██████████████████████████████| 216/216 [00:02<00:00, 83.95it/s]


Epoch 5 completed | Loss: 93.9951 | Val MRR: 0.9400

Training complete.


In [16]:
# ---- TEST SET EVALUATION ----
model.eval()
test_loader = DataLoader(TemporalLinkDataset(test_idx), batch_size=1024, shuffle=False)

test_mrr_scores = []

with torch.no_grad():
    for src, dst, ts in test_loader:
        src = src.to("cpu")
        dst = dst.to("cpu")
        ts  = ts.to("cpu")

        # negative sampling: random destination nodes
        neg_dst = torch.randint(0, num_nodes, (len(src),), device="cpu")

        pos_scores = model(src, dst, ts)
        neg_scores = model(src, neg_dst, ts)

        # MRR per sample
        scores = torch.stack([pos_scores, neg_scores], dim=1)  # (batch, 2)
        for i in range(len(scores)):
            sorted_idx = torch.argsort(scores[i], descending=True)
            rank = (sorted_idx == 0).nonzero(as_tuple=True)[0].item() + 1
            test_mrr_scores.append(1.0 / rank)

final_test_mrr = float(np.mean(test_mrr_scores))
print("Final TEST MRR:", round(final_test_mrr, 4))


Final TEST MRR: 0.9408
